In [ ]:
#@title Hindenburg Auto-Silence (automaattinen vaimennus)
#@markdown ### Vaihe 1: Suorita tämä solu
#@markdown Suorita tämä solu (Shift+Enter). Se asentaa tarvittavat lisäpaketit.
!pip install lxml

In [21]:
import os
from google.colab import drive
from ipywidgets import widgets, Dropdown, Output
from IPython.display import display

# Mount Google Drive if not already mounted
try:
    drive.mount('/content/drive', force_remount=True)
    print("\nGoogle Drive yhdistetty onnistuneesti.")
except Exception as e:
    print(f"Google Driven yhdistäminen epäonnistui: {e}")

# Define the directory in Google Drive where your files are
drive_directory = '/content/drive/MyDrive/whisper/output/'

# List files in the directory
try:
    files_in_directory = [f for f in os.listdir(drive_directory) if f.endswith('.nhsx')]
    # Prepend the directory path to the filenames for the dropdown options
    file_options = [os.path.join(drive_directory, f) for f in files_in_directory]

    print(f"\nAvailable .nhsx files in {drive_directory}:")
    if file_options:
        # Create a Dropdown widget
        file_dropdown = Dropdown(
            options=file_options,
            description='Select .nhsx file:',
            disabled=False,
        )

        # Create an output widget to display the selected value
        output = Output()

        # Function to handle dropdown value change
        def on_value_change(change):
            with output:
                # Print the selected file path
                print(f"\nSelected file path: {change['new']}")

        # Observe changes in the dropdown value
        file_dropdown.observe(on_value_change, names='value')

        # Display the dropdown and the output area
        print("\nChoose your .nhsx file from the dropdown below:")
        display(file_dropdown, output)

        # Print the initial selected file path
        if file_options:
             print(f"\nInitial selected file path: {file_dropdown.value}")


    else:
        print("No .nhsx files found in this directory to display in the dropdown.")


except FileNotFoundError:
    print(f"Error: Directory not found at {drive_directory}. Please check the path.")
except Exception as e:
    print(f"An error occurred while listing files or creating the dropdown: {e}")
    import traceback
    traceback.print_exc()

Mounted at /content/drive

Google Drive yhdistetty onnistuneesti.

Available .nhsx files in /content/drive/MyDrive/whisper/output/:

Choose your .nhsx file from the dropdown below:


Dropdown(description='Select .nhsx file:', options=('/content/drive/MyDrive/whisper/output/vst s11e06 - 2025-1…

Output()


Initial selected file path: /content/drive/MyDrive/whisper/output/vst s11e06 - 2025-10-30 litteroitu.nhsx


In [25]:
#@title Hindenburg Auto-Silence v3.1 (Korjattu versio)
#@markdown ### Vaihe 1: Yhdistä Google Drive ja aseta polku
#@markdown Suorita tämä solu (Shift+Enter). Yhdistä Google Drive pyydettäessä. **Liitä sitten `.nhsx`-tiedostosi polku alla olevaan kenttään.**
# --- Google Colab Form -kenttä ---

#@markdown ---
#@markdown ### Vaihe 2: Määritä vaimennuksen säännöt
#@markdown Säädä, miten ohjelma käsittelee hiljaisuutta. Oletusarvot perustuvat pyyntöösi.

#@markdown **Puheen "hännät" (sekunteina):** Kuinka paljon ääntä jätetään ennen puheen alkua ja sen jälkeen.
tail_duration_s = 1.0 #@param {type:"slider", min:0, max:5, step:0.1}

#@markdown **Minimaalinen hiljaisuus vaimennukselle (sekunteina):** Lyhyempiä taukoja puheen sisällä ei katkaista.
min_silence_for_muting_s = 1.0 #@param {type:"slider", min:0, max:10, step:0.1}

#@markdown **Minimaalinen klipin pituus (sekunteina):** Tätä lyhyempiä klippejä (äänellisiä tai hiljaisia) ei luoda, vaan ne yhdistetään viereiseen, jos mahdollista.
min_clip_length_s = 5.0 #@param {type:"slider", min:0, max:10, step:0.1}

# --- Ohjelman koodi alkaa tästä (ei tarvitse muokata) ---
from google.colab import drive
from lxml import etree
from collections import defaultdict
import os
import sys

# Yhdistetään Google Drive
try:
    drive.mount('/content/drive', force_remount=True)
    print("\nGoogle Drive yhdistetty onnistuneesti.")
except Exception as e:
    print(f"Google Driven yhdistäminen epäonnistui: {e}")

# --- Apufunktiot ---

def time_to_seconds(time_str):
    if not time_str: return 0.0
    try:
        if ':' in time_str:
            parts = time_str.split(':')
            seconds = float(parts[-1])
            minutes = int(parts[-2]) if len(parts) > 1 else 0
            hours = int(parts[-3]) if len(parts) > 2 else 0
            return hours * 3600 + minutes * 60 + seconds
        else:
            return float(time_str)
    except (ValueError, IndexError):
        return 0.0

def seconds_to_time(seconds):
    return f"{seconds:.3f}"

def merge_intervals_with_gap(intervals, max_gap=0.0):
    if not intervals: return []
    intervals.sort(key=lambda x: x[0])
    merged = [list(intervals[0])]
    for current_start, current_end in intervals[1:]:
        last_start, last_end = merged[-1]
        if current_start <= last_end + max_gap:
            merged[-1][1] = max(last_end, current_end)
        else:
            merged.append([current_start, current_end])
    return [tuple(i) for i in merged]

# --- Päälogiikan funktiot ---

def get_speech_intervals_for_track(tree, track_elem):
    """Hakee yhden raidan kaikki puhejaksot ja muuntaa ne aikajanalle."""
    speech_on_timeline = []
    audio_pool = tree.find('AudioPool')
    if audio_pool is None: return []

    for region in track_elem.findall('Region'):
        ref_id = region.get('Ref')
        file_elem = audio_pool.find(f".//File[@Id='{ref_id}']") # Corrected syntax error: added closing curly brace '}'
        if file_elem is None: continue

        transcription = file_elem.find('Transcription')
        if transcription is not None:
            region_start = time_to_seconds(region.get('Start', '0.0'))
            region_offset = time_to_seconds(region.get('Offset', '0.0'))
            region_length = time_to_seconds(region.get('Length', '0.0'))

            for word in transcription.findall('.//w'):
                start_str, length_str = word.get('s'), word.get('l')
                if start_str is not None and length_str is not None:
                    file_start = float(start_str)
                    file_end = file_start + float(length_str)
                    if file_start >= region_offset and file_end <= (region_offset + region_length + 0.001):
                        speech_on_timeline.append((
                            region_start + (file_start - region_offset),
                            region_start + (file_end - region_offset)
                        ))

    return sorted(speech_on_timeline)

def process_track(track_elem, all_speech_intervals):
    """Käsittelee yhden raidan: lisää hännät, tunnistaa hiljaisuudet ja pilkkoo leikkeet sääntöjen mukaan."""
    track_name = track_elem.get('Name')

    speech_groups = merge_intervals_with_gap(all_speech_intervals, min_silence_for_muting_s)
    padded_speech = [(max(0, s - tail_duration_s), e + tail_duration_s) for s, e in speech_groups]
    audible_zones = merge_intervals_with_gap(padded_speech, 0)

    original_regions = list(track_elem.findall('Region'))
    new_regions_data = []

    for region in original_regions:
        region_start = time_to_seconds(region.get('Start', '0.0'))
        region_length = time_to_seconds(region.get('Length'))
        region_end = region_start + region_length
        region_offset = time_to_seconds(region.get('Offset', '0.0'))

        cuts = {region_start, region_end}
        for zone_start, zone_end in audible_zones:
            if zone_start > region_start and zone_start < region_end: cuts.add(zone_start)
            if zone_end > region_start and zone_end < region_end: cuts.add(zone_end)

        sorted_cuts = sorted([c for c in cuts if region_start <= c <= region_end])

        for i in range(len(sorted_cuts) - 1):
            start, end = sorted_cuts[i], sorted_cuts[i+1]
            if end - start < 0.001: continue

            mid_point = start + (end - start) / 2
            is_audible = any(zone_start <= mid_point < zone_end for zone_start, zone_end in audible_zones)

            new_regions_data.append({
                'attrib': region.attrib, 'start': start, 'end': end,
                'is_audible': is_audible, 'original_offset': region_offset,
                'original_start': region_start
            })

    if not new_regions_data: return

    # Yhdistä liian lyhyet klipit älykkäästi
    final_regions_data = []
    if new_regions_data:
        final_regions_data.append(new_regions_data[0])
        for region_data in new_regions_data[1:]:
            last_region_data = final_regions_data[-1]
            current_length = region_data['end'] - region_data['start']

            # Yhdistä, jos nykyinen on liian lyhyt JA se on samaa tyyppiä kuin edellinen
            if current_length < min_clip_length_s and region_data['is_audible'] == last_region_data['is_audible']:
                last_region_data['end'] = region_data['end']
            else:
                final_regions_data.append(region_data)

    # Korvataan vanhat leikkeet uusilla
    parent = original_regions[0].getparent()
    for r in original_regions: parent.remove(r)

    for data in final_regions_data:
        new_elem = etree.Element("Region", attrib=data['attrib'])
        new_length = data['end'] - data['start']
        new_offset = data['original_offset'] + (data['start'] - data['original_start'])

        new_elem.set('Start', seconds_to_time(data['start']))
        new_elem.set('Length', seconds_to_time(new_length))
        new_elem.set('Offset', seconds_to_time(new_offset))

        if not data['is_audible']:
            new_elem.set('Muted', 'True')
        elif 'Muted' in new_elem.attrib:
            del new_elem.attrib['Muted']

        parent.append(new_elem)

    print(f"-> Raita '{track_name}' päivitetty {len(final_regions_data)} uudella leikkeellä sääntöjen mukaan.")

def run_process(input_path, output_path):
    """Koko muokkausprosessin suorittava pääfunktio."""
    if not os.path.exists(input_path) or "esimerkkitiedosto.nhsx" in input_path:
        print("\nVIRHE: Syötetiedostoa ei löytynyt tai polkua ei ole määritetty.")
        print("Kopioi polku vasemmalta ja liitä se yllä olevaan kenttään, ja suorita solu uudelleen.")
        return

    try:
        parser = etree.XMLParser(remove_blank_text=False, strip_cdata=False)
        tree = etree.parse(input_path, parser)

        tracks_elem = tree.find('Tracks')
        if tracks_elem is None:
            print("VIRHE: Tiedostosta ei löytynyt <Tracks>-osiota.")
            return

        for track_elem in tracks_elem.findall('Track'):
            track_name = track_elem.get('Name')
            print(f"\nKäsitellään raitaa: '{track_name}'...")

            speech_intervals = get_speech_intervals_for_track(tree, track_elem)
            process_track(track_elem, speech_intervals)

        print(f"\nKaikki raidat käsitelty. Kirjoitetaan muokattu tiedosto...")
        tree.write(output_path, pretty_print=True, xml_declaration=True, encoding='UTF-8')

        print("-" * 50)
        print(f"VALMIS! Muokattu tiedosto on tallennettu sijaintiin:")
        print(output_path)
        print("\nLöydät sen nyt Google Drivesta samasta kansiosta kuin alkuperäisen.")
        print("-" * 50)

    except Exception as e:
        print(f"\nKäsittelyssä tapahtui odottamaton virhe: {e}")
        import traceback
        traceback.print_exc()
        print("Varmista, että tiedosto on validi Hindenburg (.nhsx) -tiedosto ja sisältää transkriptioita.")

# --- Ohjelman suoritus ---
if 'google.colab' in sys.modules:
    base, ext = os.path.splitext(input_file_path_from_drive)
    if ext.lower() != ".nhsx":
        print(f"VAROITUS: Tiedoston pääte ei ole .nhsx. Ohjelma yrittää silti jatkaa.")

    output_file_path = f"{base}_muted{ext}"
    print(f"Syötetiedosto: {input_file_path_from_drive}")
    print(f"Tulostiedosto: {output_file_path}")
    print("-" * 50)
    run_process(input_file_path_from_drive, output_file_path)

Mounted at /content/drive

Google Drive yhdistetty onnistuneesti.
Syötetiedosto: /content/drive/MyDrive/whisper/output/vst s11e06 - 2025-10-30 base litteroitu.nhsx
Tulostiedosto: /content/drive/MyDrive/whisper/output/vst s11e06 - 2025-10-30 base litteroitu_muted.nhsx
--------------------------------------------------

Käsitellään raitaa: 'Olli'...
-> Raita 'Olli' päivitetty 323 uudella leikkeellä sääntöjen mukaan.

Käsitellään raitaa: 'Otso'...
-> Raita 'Otso' päivitetty 219 uudella leikkeellä sääntöjen mukaan.

Käsitellään raitaa: 'Kari'...
-> Raita 'Kari' päivitetty 143 uudella leikkeellä sääntöjen mukaan.

Käsitellään raitaa: 'Panu'...
-> Raita 'Panu' päivitetty 91 uudella leikkeellä sääntöjen mukaan.

Kaikki raidat käsitelty. Kirjoitetaan muokattu tiedosto...
--------------------------------------------------
VALMIS! Muokattu tiedosto on tallennettu sijaintiin:
/content/drive/MyDrive/whisper/output/vst s11e06 - 2025-10-30 base litteroitu_muted.nhsx

Löydät sen nyt Google Drivesta s